In [2]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

# Get MySQL connection details from environment variables
db_host = os.getenv('MYSQL_HOST')
db_port = os.getenv('MYSQL_PORT')
db_name_consumption = os.getenv('CONSUMPTION_DATABASE')
db_name_mart = os.getenv('MARTHOUSE_DATABASE')
db_user = os.getenv('MYSQL_USERNAME')
db_password = os.getenv('MYSQL_PASSWORD')

engine_consumption = create_engine(
    f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name_consumption}"
)

engine_mart = create_engine(
    f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name_mart}"
)

def load_table(query, engine):
    return pd.read_sql(query, engine)

In [3]:
top_cities = load_table("SELECT * FROM consumption_top10_polluting_cities", engine_consumption)
stations = load_table("SELECT * FROM dim_station", engine_mart)
hourly = load_table("SELECT * FROM consumption_city_hourly_pollution", engine_consumption)
stations_summary = load_table("SELECT * FROM mart_city_station_coverage", engine_mart)

In [4]:
import plotly.express as px

# Compute centroid per city
city_geo = stations.groupby("city_name").agg({
    "latitude": "mean",
    "longitude": "mean"
}).reset_index()

top_cities_geo = top_cities.merge(city_geo, on="city_name")

fig = px.scatter_map(
    top_cities_geo,
    lat="latitude",
    lon="longitude",
    color="pollutant_type",
    size="avg_pollution",
    hover_name="city_name",
    zoom=7,
    title="Top 10 Polluted Cities in Flanders",
    height=600
)

fig.show()

In [5]:
fig = px.bar(
    top_cities,
    x="avg_pollution",
    y="city_name",
    color="pollutant_type",
    orientation="h",
    title="Top 10 Most Polluted Cities",
    hover_data=["pollutant_type"]
)

fig.show()

In [8]:
EU_THRESHOLDS = {
    "PM10": 50,
    "PM2.5": 25,
    "NO2": 40,
    "SO2": 20
}
fig = px.bar(
    top_cities,
    x="avg_pollution",
    y="city_name",
    color="pollutant_type",
    orientation="h",
    title="Top Cities vs EU Thresholds"
)

# Add threshold lines
for pollutant, value in EU_THRESHOLDS.items():
    fig.add_vline(
        x=value,
        line_dash="dash",
        annotation_text=f"{pollutant} limit",
        annotation_position="top"
    )

fig.show()

In [10]:
city = "Antwerpen"

df_city = hourly[hourly["city_name"] == city]

fig = px.line(
    df_city,
    x="hour",
    y="avg_hourly_pollution",
    color="pollutant_type",
    markers=True,
    title=f"Hourly Pollution Pattern - {city}"
)

fig.show()

In [11]:
fig = px.scatter_map(
    stations,
    lat="latitude",
    lon="longitude",
    hover_name="station_name",
    color="city_name",
    zoom=7,
    title="Monitoring Stations Distribution",
    height=600
)

fig.show()

In [12]:
fig = px.bar(
    stations_summary,
    x="city_name",
    y="nb_stations",
    title="Number of Stations per City"
)

fig.show()

In [14]:
stations_summary = load_table("SELECT * FROM consumption_city_station_coverage", engine_consumption)

merged = top_cities.merge(stations_summary, on="city_name")

import plotly.express as px

fig = px.scatter(
    merged,
    x="nb_stations",
    y="avg_pollution",
    color="pollutant_type",
    hover_name="city_name",
    size="avg_pollution",
    title="Pollution vs Number of Stations (Interactive)"
)

fig.show()

In [18]:
import plotly.graph_objects as go

merged = top_cities.merge(
    stations_summary,
    on="city_name",
    how="inner"
)

pollutants = merged["pollutant_type"].unique()

fig = go.Figure()

for p in pollutants:
    df = merged[merged["pollutant_type"] == p]

    fig.add_trace(go.Scatter(
        x=df["nb_stations"],
        y=df["avg_pollution"],
        mode="markers+text",  # 👈 add text
        text=df["city_name"],  # 👈 city names displayed
        textposition="top center",
        name=p,
        visible=(p == pollutants[0]),
        marker=dict(size=12, opacity=0.7)
    ))

buttons = []

for i, p in enumerate(pollutants):
    visibility = [False] * len(pollutants)
    visibility[i] = True

    buttons.append(dict(
        label=p,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Pollution vs Stations — {p}"}
        ]
    ))

fig.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True
    )],
    title=f"Pollution vs Stations — {pollutants[0]}",
    xaxis_title="Number of Stations",
    yaxis_title="Average Pollution",
    height=600
)

fig.show()